Screen S&P 500 constituent stocks for the **head-and-shoulders** technical
chart pattern as of a given date and write a PDF containing one annotated
price chart per stock that exhibits the pattern.

Usage
-----
    python hs_screener.py 2024-06-14
    python hs_screener.py 2024-06-14 --cutoff 0.9 --output hs_2024-06-14.pdf
    python hs_screener.py 2024-06-14 --max-tickers 25          # quick test run
    python hs_screener.py 2024-06-14 --tickers AAPL MSFT NVDA  # specific names

Dependencies
------------
    pip install numpy scipy pandas matplotlib yfinance lxml
(lxml is used by pandas to read the live S&P 500 membership table; the PDF is
produced with matplotlib's built-in PdfPages, so no extra PDF library is
needed.)

Design overview
---------------
A generic ``Pattern`` base class holds the parameters describing a fitted
pattern.  ``PatternHeadAndShoulders`` is a concrete subclass parameterised
exactly as specified:

    parameters = d1..d7, c, b, h, y2, y6

    dot 1 = (d1, c + b*(d1 - d7))      # left  neckline point   (on line)
    dot 2 = (d2, y2)                   # left  shoulder peak     (above line)
    dot 3 = (d3, c + b*(d3 - d7))      # neckline point         (on line)
    dot 4 = (d4, y2 + h)               # head peak (the highest) (above line)
    dot 5 = (d5, c + b*(d5 - d7))      # neckline point         (on line)
    dot 6 = (d6, y6)                   # right shoulder peak     (above line)
    dot 7 = (d7, c)                    # right neckline point    (on line)

Dots 1,3,5,7 are collinear (the *neckline* price = c + b*(date - d7)); dots
2,4,6 sit above the neckline, with dot 4 the highest of the three.

``fit()`` chooses all twelve parameters from a price series so as to minimise
the vertical distance between the series and the seven connecting line
segments, subject to the structural constraints above plus the recency rule
that ``(run_date - d7) < (d7 - d5)``.

``accuracy()`` returns a distance-based quality score in [0, 1]:

    accuracy = 1 - SS_res / SS_base

where SS_res is the squared vertical distance from the price series to the
seven-segment figure (the quantity ``fit()`` minimises) and SS_base is the
squared vertical distance from the series to the figure's own neckline.  A
value of 1 means the figure passes through the data exactly (zero distance);
a value near 0 means the figure fits no better than a plain straight line,
i.e. there is no genuine head/shoulders structure.  Normalising by the
neckline (rather than by the mean, as a plain R^2 would) is what stops a flat
"figure" riding a trend from scoring highly.  Stocks scoring above the cutoff
(default 0.90) are deemed to exhibit the pattern on the run date.

Note: head-and-shoulders shapes occur readily in random price paths, so a
positive screen is a starting point for inspection, not a trading signal.
The screen can be tightened by raising ``--cutoff`` or by adding optional
criteria (e.g. shoulder-height symmetry, near-horizontal neckline) inside
``PatternHeadAndShoulders.fit``.


In [1]:
import itertools
import sys
from datetime import datetime
from typing import Optional, Sequence

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.signal import find_peaks

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

In [2]:
class Pattern:
    """Generic technical-analysis chart pattern.

    Subclasses describe a specific geometric figure as a set of *dots*
    (date/price coordinates) connected by straight line intervals.  The
    parameters stored on an instance describe the figure as fitted to the
    price series ending on the run date.
    """

    name = "Generic Pattern"

    def __init__(self) -> None:
        self.params: dict = {}      # fitted scalar parameters
        self.fitted: bool = False
        self._accuracy: float = 0.0

        self.x: Optional[np.ndarray] = None        # integer trading-day index
        self.y: Optional[np.ndarray] = None        # close prices
        self.dates: Optional[pd.DatetimeIndex] = None
        self.run_index: Optional[int] = None
        self.window: Optional[tuple[int, int]] = None  # (first_idx, last_idx) of figure

    def fit(self, prices, dates, run_index: Optional[int] = None) -> "Pattern":
        """Fit the pattern parameters to a price time series."""
        raise NotImplementedError

    def accuracy(self) -> float:
        """Fit quality in [0, 1] (1 == perfect match)."""
        return float(self._accuracy)

    def dots(self):
        """Return the figure as a list of (date, price) coordinate tuples."""
        raise NotImplementedError

    def model_curve(self, xq: np.ndarray) -> np.ndarray:
        """Piecewise-linear pattern value at index positions ``xq``."""
        raise NotImplementedError

In [3]:
class PatternHeadAndShoulders(Pattern):
    """Head-and-shoulders top, defined by 7 dots / 6 connecting segments.

    The four neckline dots (1, 3, 5, 7) lie on the straight line
    ``price = c + b * (date - d7)``.  The three intermediate dots (2, 4, 6)
    lie above that line, with dot 4 (the head) higher than dots 2 and 6
    (the shoulders).
    """

    name = "Head and Shoulders"
    # minimum number of trading days the figure must span (rejects tiny,
    # trivially-good fits)
    MIN_SPAN = 40
    # how many of the most prominent troughs to consider as neckline anchors
    MAX_TROUGH_CANDIDATES = 10

    def _neckline(self, xq):
        """Neckline price at index ``xq`` from current parameters."""
        p = self.params
        return p["c"] + p["b"] * (np.asarray(xq, float) - p["d7"])

    def dot_indices(self) -> list[int]:
        p = self.params
        return [p[f"d{k}"] for k in range(1, 8)]

    def dot_prices(self) -> list[float]:
        p = self.params
        d1, d2, d3, d4, d5, d6, d7 = self.dot_indices()
        c, b, h, y2, y6 = p["c"], p["b"], p["h"], p["y2"], p["y6"]
        return [
            c + b * (d1 - d7),   # dot 1
            y2,                  # dot 2
            c + b * (d3 - d7),   # dot 3
            y2 + h,              # dot 4 (head)
            c + b * (d5 - d7),   # dot 5
            y6,                  # dot 6
            c,                   # dot 7
        ]

    def dots(self):
        idx = self.dot_indices()
        prices = self.dot_prices()
        return [(self.dates[i], pr) for i, pr in zip(idx, prices)]

    def model_curve(self, xq: np.ndarray) -> np.ndarray:
        return np.interp(np.asarray(xq, float), self.dot_indices(), self.dot_prices())

    @staticmethod
    def _solve_prices(xw: np.ndarray, yw: np.ndarray, dx: Sequence[float]):
        """Given the seven dot date-indices ``dx`` (sorted), find the price
        parameters theta = (c, b, h, y2, y6) minimising the squared vertical
        distance between the data (xw, yw) and the seven-segment figure,
        subject to the head-and-shoulders shape constraints.

        Because every dot price is a *linear* function of theta and the
        piecewise-linear model is a linear interpolation of the dot prices,
        the objective is a convex quadratic in theta with linear inequality
        constraints -- solved here reliably with SLSQP.

        Returns (theta, sse) or (None, inf) if no feasible fit is found.
        """
        dx = np.asarray(dx, float)
        d1, d2, d3, d4, d5, d6, d7 = dx

        # Coefficient rows mapping theta = [c, b, h, y2, y6] -> each dot price.
        #            c     b          h   y2  y6
        coef = np.array([
            [1.0, d1 - d7, 0.0, 0.0, 0.0],   # p1 = c + b(d1-d7)
            [0.0, 0.0,     0.0, 1.0, 0.0],   # p2 = y2
            [1.0, d3 - d7, 0.0, 0.0, 0.0],   # p3 = c + b(d3-d7)
            [0.0, 0.0,     1.0, 1.0, 0.0],   # p4 = y2 + h
            [1.0, d5 - d7, 0.0, 0.0, 0.0],   # p5 = c + b(d5-d7)
            [0.0, 0.0,     0.0, 0.0, 1.0],   # p6 = y6
            [1.0, 0.0,     0.0, 0.0, 0.0],   # p7 = c
        ])

        # Build the design matrix A so that model = A @ theta at every data x.
        seg = np.clip(np.searchsorted(dx, xw, side="right") - 1, 0, 5)
        x0, x1 = dx[seg], dx[seg + 1]
        denom = np.where(x1 > x0, x1 - x0, 1.0)
        t = (xw - x0) / denom
        A = (1.0 - t)[:, None] * coef[seg] + t[:, None] * coef[seg + 1]

        scale = float(yw.max() - yw.min()) + 1e-9
        eps = 1e-4 * scale          # strict-inequality margin (>0)

        def obj(th):
            r = A @ th - yw
            return 0.5 * float(r @ r)

        def jac(th):
            return A.T @ (A @ th - yw)

        # Shape constraints g(theta) >= 0 (all linear -> constant Jacobians).
        cons = [
            # dot 2 strictly above neckline
            {"type": "ineq",
             "fun": lambda th: th[3] - (th[0] + th[1] * (d2 - d7)) - eps,
             "jac": lambda th: np.array([-1.0, -(d2 - d7), 0.0, 1.0, 0.0])},
            # dot 6 strictly above neckline
            {"type": "ineq",
             "fun": lambda th: th[4] - (th[0] + th[1] * (d6 - d7)) - eps,
             "jac": lambda th: np.array([-1.0, -(d6 - d7), 0.0, 0.0, 1.0])},
            # h > 0  (head above the shoulder base height y2)
            {"type": "ineq",
             "fun": lambda th: th[2] - eps,
             "jac": lambda th: np.array([0.0, 0.0, 1.0, 0.0, 0.0])},
            # dot 4 (= y2 + h) strictly above dot 6
            {"type": "ineq",
             "fun": lambda th: th[3] + th[2] - th[4] - eps,
             "jac": lambda th: np.array([0.0, 0.0, 1.0, 1.0, -1.0])},
            # dot 4 strictly above neckline
            {"type": "ineq",
             "fun": lambda th: th[3] + th[2] - (th[0] + th[1] * (d4 - d7)) - eps,
             "jac": lambda th: np.array([-1.0, -(d4 - d7), 1.0, 1.0, 0.0])},
        ]

        # Feasible-ish initial guess from the raw data.
        c0 = float(yw[-1])
        b0 = 0.0
        y2_0 = float(np.interp(d2, xw, yw))
        y6_0 = float(np.interp(d6, xw, yw))
        h0 = max(eps * 2, float(np.interp(d4, xw, yw)) - y2_0)
        th0 = np.array([c0, b0, h0, y2_0, y6_0])

        res = minimize(obj, th0, jac=jac, constraints=cons, method="SLSQP",
                       options={"maxiter": 200, "ftol": 1e-10})
        if not res.success and not np.all(np.isfinite(res.x)):
            return None, np.inf
        th = res.x
        r = A @ th - yw
        return th, float(r @ r)

    def fit(self, prices, dates, run_index: Optional[int] = None) -> "PatternHeadAndShoulders":
        y = np.asarray(prices, dtype=float)
        n = y.size
        x = np.arange(n)
        if run_index is None:
            run_index = n - 1

        self.x, self.y, self.dates, self.run_index = x, y, pd.DatetimeIndex(dates), run_index

        price_range = float(y.max() - y.min())
        if n < self.MIN_SPAN + 5 or price_range <= 0:
            self._accuracy, self.fitted = 0.0, False
            return self

        prom = 0.02 * price_range            # ignore noise smaller than 2% of range
        peaks, _ = find_peaks(y, distance=3, prominence=prom)
        troughs, tprops = find_peaks(-y, distance=3, prominence=prom)

        if peaks.size < 3 or troughs.size < 2:
            self._accuracy, self.fitted = 0.0, False
            return self

        # Keep the most prominent troughs, and add the series endpoints as
        # possible neckline anchors (d1 near the start, d7 near the end).
        order = np.argsort(tprops["prominences"])[::-1]
        trough_cands = list(troughs[order][: self.MAX_TROUGH_CANDIDATES])
        trough_cands += [0, n - 1]
        trough_cands = sorted(set(int(t) for t in trough_cands))

        def best_peak(lo: int, hi: int) -> Optional[int]:
            inside = peaks[(peaks > lo) & (peaks < hi)]
            if inside.size == 0:
                return None
            return int(inside[np.argmax(y[inside])])

        best = None  # (accuracy, dx, theta, sse)

        for d1, d3, d5, d7 in itertools.combinations(trough_cands, 4):
            # recency rule: (run_date - d7) must be shorter than (d5 -> d7)
            if (run_index - d7) >= (d7 - d5):
                continue
            if (d7 - d1) < self.MIN_SPAN:
                continue

            d2 = best_peak(d1, d3)
            d4 = best_peak(d3, d5)
            d6 = best_peak(d5, d7)
            if d2 is None or d4 is None or d6 is None:
                continue

            dx = (d1, d2, d3, d4, d5, d6, d7)
            xw = np.arange(d1, d7 + 1)
            yw = y[d1: d7 + 1]

            theta, sse = self._solve_prices(xw, yw, dx)
            if theta is None:
                continue

            # Accuracy = how much of the data's squared vertical deviation from
            # the (fitted) neckline is removed by the full seven-segment figure.
            #   SS_res  : squared distance from data to the H&S figure  (== sse)
            #   SS_base : squared distance from data to the bare neckline
            # accuracy = 1 - SS_res/SS_base  (clamped to [0, 1]).
            #   * 1  -> the figure passes through the data (zero distance);
            #   * ~0 -> the figure fits no better than a plain line, i.e. the
            #           head/shoulders structure is absent (e.g. a trend).
            # This makes the metric distance-based yet immune to the degenerate
            # "flat figure riding a trend" fit that a plain R^2 would reward.
            
            c, b = float(theta[0]), float(theta[1])
            neck = c + b * (xw - dx[6])
            ss_base = float(((yw - neck) ** 2).sum())
            scale = float(np.median(yw))
            if ss_base <= 1e-8 * max(1.0, scale * scale) * yw.size:
                continue                       # data already lies on a line
            acc = max(0.0, min(1.0, 1.0 - sse / ss_base))

            if best is None or acc > best[0]:
                best = (acc, dx, theta, sse)

        if best is None:
            self._accuracy, self.fitted = 0.0, False
            return self

        acc, dx, theta, sse = best
        c, b, h, y2, y6 = (float(v) for v in theta)
        self.params = {
            "d1": int(dx[0]), "d2": int(dx[1]), "d3": int(dx[2]), "d4": int(dx[3]),
            "d5": int(dx[4]), "d6": int(dx[5]), "d7": int(dx[6]),
            "c": c, "b": b, "h": h, "y2": y2, "y6": y6,
            # convenience: dot dates recalibrated to the latest series
            "dot_dates": [self.dates[i] for i in dx],
            "sse": sse,
        }
        self.window = (int(dx[0]), int(dx[6]))
        self._accuracy = acc
        self.fitted = True
        return self

In [5]:
def fetch_price_history(tickers: Sequence[str], as_of: pd.Timestamp,
                        lookback: int = 250, chunk: int = 40) -> dict[str, pd.Series]:
    import yfinance as yf
    as_of = pd.Timestamp(as_of)
    end = (as_of + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    # ~1.8 calendar days per trading day, plus a buffer, to be sure of coverage
    start = (as_of - pd.Timedelta(days=int(lookback * 1.8) + 40)).strftime("%Y-%m-%d")

    out: dict[str, pd.Series] = {}
    tickers = list(tickers)
    for i in range(0, len(tickers), chunk):
        batch = tickers[i: i + chunk]
        try:
            df = yf.download(batch, start=start, end=end, auto_adjust=True,
                             progress=False, group_by="ticker", threads=True)
        except Exception as exc:
            print(f"[warn] download failed for batch {i//chunk} ({exc})",
                  file=sys.stderr)
            continue

        for t in batch:
            try:
                if len(batch) == 1:
                    s = df["Close"]
                else:
                    s = df[t]["Close"]
                s = pd.Series(s).dropna()
                s = s[s.index <= as_of]
                if len(s) >= max(60, int(lookback * 0.6)):
                    out[t] = s.tail(lookback)
            except Exception:
                continue
        print(f"  fetched {len(out)} / {len(tickers)} ...", end="\r", file=sys.stderr)
    print(file=sys.stderr)
    return out

In [6]:
def plot_pattern(ticker: str, pat: PatternHeadAndShoulders, pdf: PdfPages) -> None:
    """Render one annotated chart for ``ticker`` and append it to ``pdf``."""
    dates, y = pat.dates, pat.y
    dot_idx = pat.dot_indices()
    dot_prices = pat.dot_prices()
    dot_dates = [dates[i] for i in dot_idx]

    fig, ax = plt.subplots(figsize=(11, 6.2))
    ax.plot(dates, y, color="0.45", lw=1.1, label="Adj. close")

    # neckline (line through dots 1,3,5,7), drawn across the figure span
    d1, d7 = dot_idx[0], dot_idx[-1]
    neck_x = [dates[d1], dates[d7]]
    neck_y = [pat._neckline(d1), pat._neckline(d7)]
    ax.plot(neck_x, neck_y, "--", color="#1f4e79", lw=1.4, label="neckline")

    # the seven connecting segments and the dots
    ax.plot(dot_dates, dot_prices, "-", color="#c0392b", lw=2.0, zorder=4,
            label="H&S figure")
    ax.plot(dot_dates, dot_prices, "o", color="#c0392b", ms=7, zorder=5)
    for k, (dd, pp) in enumerate(zip(dot_dates, dot_prices), start=1):
        ax.annotate(str(k), (dd, pp), textcoords="offset points",
                    xytext=(0, 9 if k % 2 == 0 else -14), ha="center",
                    fontsize=9, fontweight="bold", color="#7b241c")

    ax.set_title(f"{ticker}    Head & Shoulders    "
                 f"(accuracy = {pat.accuracy():.1%})", fontsize=13)
    ax.set_ylabel("Price")
    ax.grid(True, alpha=0.25)
    ax.legend(loc="best", fontsize=9)
    fig.autofmt_xdate()
    fig.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)

In [7]:
as_of_ts = pd.Timestamp("20260623")
nLookback = 250
nCutoff = .9
sOutput = f"head_and_shoulders_{as_of_ts.date()}.pdf"

In [10]:
import io
from curl_cffi import requests

session = requests.Session(impersonate="chrome")
wikiPg = session.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies").text
tickerList = pd.read_html(io.StringIO(wikiPg))[0]["Symbol"].tolist()
len(tickerList)
#        tables = pd.read_html("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies")
#        symbols = tables[0]["Symbol"].astype(str).tolist()
#        tickers = [s.strip().replace(".", "-") for s in symbols if s and s != "nan"]
#        len(tickers)

503

In [11]:
history = fetch_price_history(tickerList, as_of_ts, lookback=nLookback)
len(history)

  fetched 40 / 503 ...
2 Failed downloads:
['BF.B']: YFPricesMissingError('possibly delisted; no price data found  (1d 2025-02-18 -> 2026-06-24)')
['BRK.B']: YFTzMissingError('possibly delisted; no timezone found')
  fetched 500 / 503 ...


500

In [15]:
    hits: list[tuple[str, PatternHeadAndShoulders]] = []
    for j, (ticker, series) in enumerate(sorted(history.items()), start=1):
        try:
            pat = PatternHeadAndShoulders().fit(series.values, series.index)
        except Exception as exc:
            print(f"[warn] fit failed for {ticker}: {exc}")
            continue
        if pat.fitted and pat.accuracy() >= nCutoff:
            hits.append((ticker, pat))
            print(f"{ticker}: success")
        else:
            print(f"{ticker}: not detected, {pat.fitted} : {pat.accuracy()}")

A: success
AAPL: success
ABBV: success
ABNB: success
ABT: not detected, True : 0.8751663295356541
ACGL: not detected, True : 0.8718699094526849
ACN: success
ADBE: success
ADI: success
ADM: not detected, True : 0.8505782958924878
ADP: not detected, True : 0.8275756767475654
ADSK: success
AEE: success
AEP: success
AES: not detected, True : 0.7887487246716327
AFL: success
AIG: success
AIZ: success
AJG: not detected, True : 0.8633654789096606
AKAM: not detected, True : 0.777504864585588
ALB: not detected, True : 0.8923855020764062
ALGN: success
ALL: not detected, True : 0.7791658933440527
ALLE: not detected, True : 0.8998836635820618
AMAT: not detected, True : 0.807823843743161
AMCR: success
AMD: not detected, True : 0.893154548759158
AME: success
AMGN: success
AMP: not detected, True : 0.8952456904187873
AMT: not detected, True : 0.8758571468019679
AMZN: not detected, True : 0.8426175113571619
ANET: success
AON: not detected, True : 0.8173553943452359
AOS: success
APA: success
APD: succes

In [16]:
    hits.sort(key=lambda kv: kv[1].accuracy(), reverse=True)
    with PdfPages(sOutput) as pdf:
        if not hits:
            fig, ax = plt.subplots(figsize=(11, 6))
            ax.axis("off")
            ax.text(0.5, 0.5,
                    f"No stocks exhibited a head-and-shoulders pattern\n"
                    f"above the {nCutoff:.0%} accuracy cutoff as of "
                    f"{as_of_ts.date()}.",
                    ha="center", va="center", fontsize=14)
            pdf.savefig(fig)
            plt.close(fig)
        else:
            for ticker, pat in hits:
                    dates, y = pat.dates, pat.y
                    dot_idx = pat.dot_indices()
                    dot_prices = pat.dot_prices()
                    dot_dates = [dates[i] for i in dot_idx]
                
                    fig, ax = plt.subplots(figsize=(11, 6.2))
                    ax.plot(dates, y, color="0.45", lw=1.1, label="Adj. close")
                
                    d1, d7 = dot_idx[0], dot_idx[-1]
                    neck_x = [dates[d1], dates[d7]]
                    neck_y = [pat._neckline(d1), pat._neckline(d7)]
                    ax.plot(neck_x, neck_y, "--", color="#1f4e79", lw=1.4, label="neckline")
                
                    ax.plot(dot_dates, dot_prices, "-", color="#c0392b", lw=2.0, zorder=4, label="H&S figure")
                    ax.plot(dot_dates, dot_prices, "o", color="#c0392b", ms=7, zorder=5)
                    for k, (dd, pp) in enumerate(zip(dot_dates, dot_prices), start=1):
                        ax.annotate(str(k), (dd, pp), textcoords="offset points",
                                    xytext=(0, 9 if k % 2 == 0 else -14), ha="center", fontsize=9, fontweight="bold", color="#7b241c")

                    ax.set_title(f"{ticker}    Head & Shoulders    (accuracy = {pat.accuracy():.1%})", fontsize=13)
                    ax.set_ylabel("Price")
                    ax.grid(True, alpha=0.25)
                    ax.legend(loc="best", fontsize=9)
                    fig.autofmt_xdate()
                    fig.tight_layout()
                    pdf.savefig(fig)
                    plt.close(fig)

        d = pdf.infodict()
        d["Title"] = f"S&P 500 Head-and-Shoulders screen {as_of_ts.date()}"
        d["CreationDate"] = datetime.now()